# Notebook 5: Regression Models — Predicting Exact FED Rate
## Federal Reserve Interest Rate Prediction

**Models:** Linear Regression, Ridge, Lasso, SVR, Decision Tree, Random Forest, Gradient Boosting, XGBoost

**Key concepts demonstrated:**
- Regularization (Ridge L2, Lasso L1)
- Hyperparameter tuning
- TimeSeriesSplit cross-validation
- Overfitting/underfitting analysis via learning curves
- Feature importance and coefficient analysis


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, TimeSeriesSplit)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb


In [ ]:
# Load preprocessed data
import pickle
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)

X_scaled  = data['X_scaled']
y_reg     = data['y_reg']
feat_cols = data['feature_cols']

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_reg, test_size=0.2, random_state=42)
tscv = TimeSeriesSplit(n_splits=5)

print(f"Training set: {X_tr.shape}")
print(f"Test set:     {X_te.shape}")
print(f"Target range: {y_reg.min():.2f}% to {y_reg.max():.2f}%")


In [ ]:
def evaluate(model, name, X_tr=X_tr, X_te=X_te, y_tr=y_tr, y_te=y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    cv_r2 = cross_val_score(model, X_scaled, y_reg, cv=tscv, scoring='r2')
    print(f"{name}: RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  CV R²={cv_r2.mean():.4f}±{cv_r2.std():.4f}")
    return y_pred, {'RMSE':rmse,'MAE':mae,'R2':r2,'CV':cv_r2.mean()}

results = {}


## 1. Linear Regression — Baseline

In [ ]:
lr = LinearRegression()
y_pred_lr, results['Linear'] = evaluate(lr, "Linear Regression")

# Coefficients
coef_df = pd.Series(lr.coef_, index=feat_cols).sort_values(key=abs, ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top20 = coef_df.head(20)
axes[0].barh(range(20), top20.values,
             color=['#F44336' if v < 0 else '#2196F3' for v in top20.values])
axes[0].set_yticks(range(20)); axes[0].set_yticklabels(top20.index, fontsize=8)
axes[0].axvline(0, color='k', linewidth=0.8)
axes[0].set_title('Linear Reg — Top 20 Coefficients', fontweight='bold')
axes[1].scatter(y_te, y_pred_lr, alpha=0.4, s=20, color='#2196F3')
mn, mx = min(y_te.min(), y_pred_lr.min()), max(y_te.max(), y_pred_lr.max())
axes[1].plot([mn,mx],[mn,mx],'r--', linewidth=2)
axes[1].set_xlabel('Actual'); axes[1].set_ylabel('Predicted')
axes[1].set_title(f"Linear Reg: R²={results['Linear']['R2']:.4f}", fontweight='bold')
plt.tight_layout(); plt.show()


## 2. Ridge & Lasso — Regularization

In [ ]:
# Ridge — hyperparameter tuning
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_scores = [cross_val_score(Ridge(alpha=a), X_scaled, y_reg, cv=tscv, scoring='r2').mean()
                for a in alphas]
best_alpha_ridge = alphas[np.argmax(ridge_scores)]

ridge = Ridge(alpha=best_alpha_ridge)
y_pred_ridge, results['Ridge'] = evaluate(ridge, f"Ridge (alpha={best_alpha_ridge})")

# Lasso — hyperparameter tuning
lasso_scores = [cross_val_score(Lasso(alpha=a, max_iter=10000), X_scaled, y_reg,
                                cv=tscv, scoring='r2').mean() for a in alphas]
best_alpha_lasso = alphas[np.argmax(lasso_scores)]

lasso = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso.fit(X_tr, y_tr)
y_pred_lasso, results['Lasso'] = evaluate(lasso, f"Lasso (alpha={best_alpha_lasso})")

n_zero = (lasso.coef_ == 0).sum()
print(f"\nLasso zeroed out {n_zero}/{len(lasso.coef_)} features ({n_zero/len(lasso.coef_)*100:.0f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].semilogx(alphas, ridge_scores, 'b-o', label='Ridge')
axes[0].semilogx(alphas, lasso_scores, 'r-o', label='Lasso')
axes[0].axvline(best_alpha_ridge, color='blue', linestyle='--', alpha=0.7)
axes[0].axvline(best_alpha_lasso, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Alpha'); axes[0].set_ylabel('CV R²')
axes[0].set_title('Ridge vs Lasso: Alpha Tuning', fontweight='bold'); axes[0].legend()

non_zero = np.where(lasso.coef_ != 0)[0]
if len(non_zero) > 0:
    top_lasso = non_zero[np.argsort(np.abs(lasso.coef_[non_zero]))[-15:]]
    axes[1].barh(range(len(top_lasso)), lasso.coef_[top_lasso],
                 color=['#F44336' if v < 0 else '#2196F3' for v in lasso.coef_[top_lasso]])
    axes[1].set_yticks(range(len(top_lasso)))
    axes[1].set_yticklabels([feat_cols[i][:20] for i in top_lasso], fontsize=8)
    axes[1].axvline(0, color='k', linewidth=0.8)
axes[1].set_title(f"Lasso Non-Zero Coefficients ({n_zero} zeroed)", fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Decision Tree — Depth Tuning & Overfitting

In [ ]:
# Depth-based overfitting analysis
depths = range(1, 20)
train_r2s, val_r2s = [], []
for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_r2s.append(r2_score(y_tr, dt.predict(X_tr)))
    val_r2s.append(cross_val_score(dt, X_scaled, y_reg, cv=tscv, scoring='r2').mean())

best_depth = depths[np.argmax(val_r2s)]
print(f"Best depth (TimeSeriesCV): {best_depth}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_r2s, 'b-o', label='Train R²')
ax.plot(depths, val_r2s, 'r-o', label='TimeSeriesCV R²')
ax.axvline(best_depth, color='green', linestyle='--', label=f'Best depth={best_depth}')
ax.fill_between(depths, train_r2s, val_r2s, alpha=0.15, color='purple', label='Overfitting Gap')
ax.set_xlabel('Tree Depth'); ax.set_ylabel('R²')
ax.set_title('Decision Tree: Bias-Variance Tradeoff', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()

dt = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
_, results['Decision Tree'] = evaluate(dt, f"Decision Tree (depth={best_depth})")


## 4. Ensemble Models — RF, GBM, XGBoost

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=5,
                            random_state=42, n_jobs=-1)
y_pred_rf, results['Random Forest'] = evaluate(rf, "Random Forest")
rf.fit(X_tr, y_tr)

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
_, results['Gradient Boosting'] = evaluate(gb, "Gradient Boosting")

# XGBoost
xgb_reg = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
y_pred_xgb, results['XGBoost'] = evaluate(xgb_reg, "XGBoost")
xgb_reg.fit(X_tr, y_tr)

# Feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
rf_fi  = pd.Series(rf.feature_importances_,  index=feat_cols).sort_values(ascending=False).head(15)
xgb_fi = pd.Series(xgb_reg.feature_importances_, index=feat_cols).sort_values(ascending=False).head(15)
rf_fi.plot.bar(ax=axes[0], color=PALETTE[:15])
axes[0].set_title('Random Forest Feature Importance', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
xgb_fi.plot.bar(ax=axes[1], color=PALETTE[:15])
axes[1].set_title('XGBoost Feature Importance', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


## 5. Learning Curves — Overfitting/Underfitting

In [ ]:
def plot_lc(model, title):
    ts, tr, vl = learning_curve(model, X_scaled, y_reg, cv=tscv,
                                scoring='r2', train_sizes=np.linspace(0.1,1.0,10))
    tr_m, tr_s = tr.mean(1), tr.std(1)
    vl_m, vl_s = vl.mean(1), vl.std(1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(ts, tr_m, 'b-o', label='Train R²')
    ax.fill_between(ts, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color='blue')
    ax.plot(ts, vl_m, 'r-o', label='Validation R²')
    ax.fill_between(ts, vl_m-vl_s, vl_m+vl_s, alpha=0.15, color='red')
    gap = tr_m[-1] - vl_m[-1]
    status = 'OVERFITTING' if gap > 0.1 else ('UNDERFITTING' if vl_m[-1] < 0.5 else 'GOOD FIT')
    ax.set_title(f'{title} — Learning Curve\nStatus: {status} (gap={gap:.3f})', fontweight='bold')
    ax.set_xlabel('Training Size'); ax.set_ylabel('R²'); ax.legend()
    plt.tight_layout(); plt.show()

for model, name in [(xgb_reg,'XGBoost'),(rf,'Random Forest'),(lasso,'Lasso')]:
    plot_lc(model, name)


## 6. Final Comparison

In [ ]:
# Results table
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df.sort_values('R2', ascending=False)
display(results_df.round(4).style.highlight_max(subset=['R2'], color='lightgreen')
                                  .highlight_min(subset=['RMSE'], color='lightgreen'))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models = results_df.index
axes[0].bar(models, results_df['R2'], color=PALETTE[:len(models)])
axes[0].set_xticklabels(models, rotation=35, ha='right')
axes[0].set_title('R² Score Comparison', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[1].bar(models, results_df['RMSE'], color=PALETTE[:len(models)])
axes[1].set_xticklabels(models, rotation=35, ha='right')
axes[1].set_title('RMSE Comparison (lower=better)', fontweight='bold')
plt.tight_layout(); plt.show()
